# Lab 02 — The Model, Built From Small Pieces
## From mathematical operations to a causal Transformer
**Language:** English · **Duration:** two sessions of 90–120 minutes · **Hardware:** CPU

This independently authored lab follows every subsection of Step 2 in FareedKhan-dev's repository. Its instructional approach draws on Stanford CS336's emphasis on implementing language-model components and validating their behavior. It is not an official Stanford assignment.

**Readings**
- [Repository Step 2](https://github.com/FareedKhan-dev/train-llm-from-scratch#step-2-the-model-built-from-small-pieces)
- [Stanford CS336, Spring 2025](https://cs336.stanford.edu/spring2025/)
- [Stanford Assignment 1](https://github.com/stanford-cs336/assignment1-basics)
- [Repository modern architecture](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/docs/modern/README.md)

The classic architecture is implemented completely below. The modern section develops each Step 2 replacement with equations and small executable experiments. Follow its source links for the repository's complete modern model and cache infrastructure.

**Learning outcomes**
1. Trace every tensor dimension from IDs to vocabulary logits.
2. Derive scaled dot-product attention and its causal mask.
3. Assemble MLP, attention heads, pre-norm residual blocks, and a full model.
4. Verify causality, gradient flow, and next-token loss.
5. Explain RMSNorm, RoPE, SwiGLU, GQA, MLA, efficient attention, caching, and MoE.

**How to study:** For each section, write the expected shape, work the short calculation, run the implementation, then change one input and explain the result. Complete the exercises before reading the reference cells.

**Setup:** Use a Python environment with PyTorch installed (`python -m pip install torch`). In a notebook, `%pip install torch` installs into the active kernel. Restart the kernel if needed, then run cells in order. No model or corpus downloads are required. Code was syntax-checked when this notebook was prepared; PyTorch execution was unavailable in the authoring environment.

In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42)
torch.set_num_threads(1)
B, T, C, H = 2, 8, 32, 4
assert C % H == 0
print(torch.__version__)

## 1. Recover the contract from Lab 1
Lab 1 produces integer inputs and targets of shape $(B,T)$. A model maps inputs to logits of shape $(B,T,V)$, where $V$ is vocabulary size. Each vector of $V$ logits scores the next token at one position.

| Symbol | Meaning | Example |
|---|---|---|
| $B$ | batch size | 2 |
| $T$ | sequence length | 8 |
| $C$ | embedding width | 32 |
| $H$ | attention heads | 4 |
| $d=C/H$ | width per head | 8 |
| $L$ | number of blocks | 2 |

The following English corpus uses a character vocabulary, matching the introductory approach in Lab 1. For an existing Lab 1 batch, keep its tokenizer mapping and set $V$ to cover all IDs. IDs must be integer indices into the embedding table.

In [ ]:
text = "The cat sat by the window. The dog waited by the door. "
characters = sorted(set(text))
stoi = {character: index for index, character in enumerate(characters)}
V = len(characters)
stream = torch.tensor([stoi[character] for character in text], dtype=torch.long)
chunks = torch.stack([stream[start:start + T + 1] for start in (0, 12)])
inputs, targets = chunks[:, :-1], chunks[:, 1:]
assert inputs.shape == targets.shape == (B, T)
assert torch.equal(inputs[:, 1:], targets[:, :-1])

## 2. Embeddings and position information
An embedding table $E\in\mathbb{R}^{V\times C}$ contains a learned row for each token. A position table $P\in\mathbb{R}^{T_{max}\times C}$ contains a learned row for each position.

$$X_{b,t,:}=E[\mathrm{ids}_{b,t}]+P[t].$$

Looking up a row is equivalent to multiplying a one-hot token vector by $E$, but avoids building the one-hot tensor. Position vectors give the classic model an explicit representation of order.

**Exercise:** If $V=30,C=32,T_{max}=16$, how many parameters do these two tables contain? Answer: $30(32)+16(32)=1472$.

In [ ]:
token_embedding = nn.Embedding(V, C)
position_embedding = nn.Embedding(16, C)
x = token_embedding(inputs) + position_embedding(torch.arange(T))
assert x.shape == (B, T, C)
one_hot = F.one_hot(inputs, num_classes=V).float()
assert torch.allclose(one_hot @ token_embedding.weight, token_embedding(inputs))

## 3. Multi-layer perceptron
The repository expands each token vector from $C$ to $4C$, applies ReLU, and projects back.

$$\mathrm{MLP}(X)=\mathrm{ReLU}(XW_1+b_1)W_2+b_2.$$

$W_1$ has mathematical shape $(C,4C)$ and $W_2$ has shape $(4C,C)$. PyTorch stores linear weights transposed, as `(out_features, in_features)`. ReLU applies $\max(0,z)$ elementwise. The same weights operate independently at every batch and sequence position.

**Exercise:** Derive the parameter count including biases. Then change a single input token and check which output positions change. Expected count: $8C^2+5C$.

[Repository MLP](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/src/models/mlp.py)

In [ ]:
class MLP(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.hidden = nn.Linear(width, 4 * width)
        self.proj = nn.Linear(4 * width, width)

    def forward(self, x):
        return self.proj(F.relu(self.hidden(x)))

mlp = MLP(C)
assert sum(p.numel() for p in mlp.parameters()) == 8 * C * C + 5 * C
assert mlp(x).shape == x.shape
changed = x.detach().clone()
changed[:, 3] += 1
assert torch.allclose(mlp(x)[:, :3], mlp(changed)[:, :3])

## 4. Single-head attention
Project the input into queries, keys, and values.

$$Q=XW_Q,\quad K=XW_K,\quad V_h=XW_V.$$
$$S_{i,j}=\frac{q_i^T k_j}{\sqrt d},\qquad A=\mathrm{softmax}(S+M),\qquad O=AV_h.$$

| Tensor | Shape | Interpretation |
|---|---|---|
| $Q,K,V_h$ | $(B,T,d)$ | per-token vectors |
| $S,A$ | $(B,T,T)$ | query-to-key scores and weights |
| $O$ | $(B,T,d)$ | weighted value vectors |

If query and key coordinates are independent with mean zero and variance one, the variance of their dot product is $d$. Division by $\sqrt d$ controls that initial scale. Softmax runs over the key dimension so each query's weights sum to one.

For causal attention, $M_{i,j}=0$ when $j\le i$ and $-\infty$ otherwise. A future position receives probability zero. Mask before softmax.

**Paper exercise:** Let all query and key vectors be zero and let scalar values be $[2,4,9]$. The causal outputs must be $[2,3,5]$. Explain why the first row has exactly one allowed key.

[Repository attention](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/src/models/attention.py)

In [ ]:
def causal_attention(q, k, v):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    length = q.shape[-2]
    allowed = torch.ones(length, length, dtype=torch.bool, device=q.device).tril()
    weights = scores.masked_fill(~allowed, float("-inf")).softmax(dim=-1)
    return weights @ v, weights

zero = torch.zeros(1, 3, 1)
out, weights = causal_attention(zero, zero, torch.tensor([[[2.], [4.], [9.]]]))
assert torch.allclose(out.flatten(), torch.tensor([2., 3., 5.]))
assert torch.allclose(weights.sum(-1), torch.ones(1, 3))
assert torch.count_nonzero(weights.triu(1)) == 0
print(weights)

class Head(nn.Module):
    def __init__(self, width, head_width):
        super().__init__()
        self.query = nn.Linear(width, head_width, bias=False)
        self.key = nn.Linear(width, head_width, bias=False)
        self.value = nn.Linear(width, head_width, bias=False)

    def forward(self, x):
        output, _ = causal_attention(self.query(x), self.key(x), self.value(x))
        return output

assert Head(C, C // H)(x).shape == (B, T, C // H)

## 5. Multi-head attention
Each head has separate learned projections. Concatenate along the feature axis and mix with an output projection.

$$\mathrm{MHA}(X)=\mathrm{Concat}(O_1,\ldots,O_H)W_O+b_O.$$

Each $O_h$ has width $d=C/H$, so concatenation restores width $C$. Head specialization is learned; specific linguistic roles are not guaranteed.

**Exercise:** Predict the failure if you concatenate over the sequence axis. Derive the parameter count with bias-free Q/K/V and a biased output projection. Answer: $4C^2+C$.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        if width % heads:
            raise ValueError("Embedding width must be divisible by head count.")
        self.heads = nn.ModuleList([Head(width, width // heads) for _ in range(heads)])
        self.proj = nn.Linear(width, width)

    def forward(self, x):
        return self.proj(torch.cat([head(x) for head in self.heads], dim=-1))

attention = MultiHeadAttention(C, H)
assert attention(x).shape == x.shape
assert sum(p.numel() for p in attention.parameters()) == 4 * C * C + C

## 6. Pre-norm residual blocks
LayerNorm computes statistics over the feature axis of each token.

$$\mathrm{LN}(x)=\gamma\odot\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta.$$
$$U=X+\mathrm{MHA}(\mathrm{LN}_1(X)),\qquad Y=U+\mathrm{MLP}(\mathrm{LN}_2(U)).$$

Residual addition requires matching shapes. For $y=x+f(x)$, the Jacobian is $I+J_f$, giving gradients a direct identity path. This helps optimization without guaranteeing that every deep model trains stably.

**Exercise:** Zero the attention and MLP parameters. What should the entire block return? Verify the identity numerically.

[Repository block](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/src/models/transformer_block.py)

In [ ]:
class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attn = MultiHeadAttention(width, heads)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = MLP(width)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

identity_block = Block(C, H)
with torch.no_grad():
    for module in (identity_block.attn, identity_block.mlp):
        for parameter in module.parameters():
            parameter.zero_()
assert torch.allclose(identity_block(x), x)

## 7. Full model and cross-entropy
Stack $L$ blocks, apply final LayerNorm, then project to vocabulary scores.

$$Z=\mathrm{LN}_{final}(X_L)W_{vocab}+b_{vocab}.$$
$$\mathcal L=-\frac1{BT}\sum_{b,t}\log\frac{\exp Z_{b,t,y_{b,t}}}{\sum_{v=1}^{V}\exp Z_{b,t,v}}.$$

Pass logits directly to cross-entropy, which computes log-softmax internally. Flatten the batch and sequence dimensions with `reshape`. A sliced target tensor may be non-contiguous; `reshape` handles that layout.

`forward_hidden` exposes normalized hidden states for later reward/value heads. This educational implementation preserves the classic Step 2 computations while building its attention mask dynamically. The repository also provides generation and optional gradient checkpointing.

[Repository model](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/src/models/transformer.py)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, width, heads, context_length, layers):
        super().__init__()
        self.context_length = context_length
        self.token_embed = nn.Embedding(vocab_size, width)
        self.position_embed = nn.Embedding(context_length, width)
        self.blocks = nn.ModuleList([Block(width, heads) for _ in range(layers)])
        self.norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size)

    def forward_hidden(self, ids):
        length = ids.shape[1]
        if not 1 <= length <= self.context_length:
            raise ValueError("Sequence length is outside the supported context.")
        positions = torch.arange(length, device=ids.device)
        hidden = self.token_embed(ids) + self.position_embed(positions)
        for block in self.blocks:
            hidden = block(hidden)
        return self.norm(hidden)

    def forward(self, ids, targets=None):
        logits = self.lm_head(self.forward_hidden(ids))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1).long())
        return logits, loss

model = Transformer(V, C, H, 16, 2)
logits, loss = model(inputs, targets)
assert logits.shape == (B, T, V)
assert loss.ndim == 0 and torch.isfinite(loss)
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
print("Loss:", loss.item(), "Parameters:", sum(p.numel() for p in model.parameters()))

## 8. Verify behavior, not just shapes
A future token must never affect earlier logits. This test detects a missing or reversed causal mask even when all tensor dimensions look correct.

For the classic implementation here, including both independent vocabulary matrices and biases, the total parameter count is
$$2VC+T_{max}C+L(12C^2+10C)+2C+V.$$
Large vocabularies can dominate a small model's parameter budget. Compute counts from the actual configuration rather than copying a model-size label.

**Exercise:** Remove the causal mask temporarily. Run the test and explain the failure. Restore it afterward.

In [ ]:
model.eval()
modified = inputs.clone()
modified[:, 5:] = (modified[:, 5:] + 1) % V
with torch.no_grad():
    original_logits, _ = model(inputs)
    modified_logits, _ = model(modified)
assert torch.allclose(original_logits[:, :5], modified_logits[:, :5], atol=1e-6)
expected_count = 2 * V * C + 16 * C + 2 * (12 * C * C + 10 * C) + 2 * C + V
assert sum(p.numel() for p in model.parameters()) == expected_count
print("Causality and parameter-count checks passed.")

## 9. Modern replacements — RMSNorm and SwiGLU
The repository's modern model retains the residual skeleton.

**RMSNorm** removes mean subtraction and learns a scale:
$$\mathrm{RMSNorm}(x)=g\odot x/\sqrt{\mathrm{mean}(x^2)+\epsilon}.$$

**SwiGLU** multiplies a smooth gate with a second projection:
$$\mathrm{SwiGLU}(x)=\big[\mathrm{SiLU}(xW_g)\odot(xW_u)\big]W_d,$$
where $\mathrm{SiLU}(z)=z\sigma(z)$. Three bias-free matrices use approximately $3CF$ parameters. Setting $F\approx8C/3$ matches a classic $8C^2$ MLP budget, before implementation-specific rounding.

**Exercise:** Compare the output mean of LayerNorm and RMSNorm on vectors with a positive offset. Explain why only LayerNorm explicitly subtracts the mean.

[Normalization, MLP, tying, and initialization](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/docs/modern/blocks.md)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, width, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.square().mean(-1, keepdim=True) + self.eps) * self.scale

class SwiGLU(nn.Module):
    def __init__(self, width, hidden_width):
        super().__init__()
        self.gate = nn.Linear(width, hidden_width, bias=False)
        self.up = nn.Linear(width, hidden_width, bias=False)
        self.down = nn.Linear(hidden_width, width, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

assert RMSNorm(C)(x).shape == x.shape
assert SwiGLU(C, 88)(x).shape == x.shape

## 10. Rotary position embeddings
RoPE rotates pairs of query/key coordinates at each position. For pair index $i$, use $\theta_i=10000^{-2i/d}$ and rotate by $p\theta_i$.

$$(a,b) \mapsto (a\cos\phi-b\sin\phi,\ a\sin\phi+b\cos\phi).$$

The identity $(R_pq)^T(R_sk)=q^TR_{s-p}k$ makes the positional effect depend on relative displacement. Apply rotations to Q and K inside attention. Our demonstration uses adjacent coordinate pairs; implementations can choose other consistent pairing conventions.

**Exercise:** Shift both positions by 7 and check that the dot product remains unchanged.

[Repository RoPE explanation](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/docs/modern/rope.md)

In [ ]:
def rotate_vector(vector, position):
    width = vector.shape[-1]
    assert width % 2 == 0
    frequencies = 10000.0 ** (-torch.arange(0, width, 2, device=vector.device).float() / width)
    angle = position * frequencies
    pairs = vector.reshape(-1, 2)
    a, b = pairs[:, 0], pairs[:, 1]
    return torch.stack((a * angle.cos() - b * angle.sin(), a * angle.sin() + b * angle.cos()), -1).flatten()

q, k = torch.randn(8), torch.randn(8)
score = rotate_vector(q, 2) @ rotate_vector(k, 5)
shifted_score = rotate_vector(q, 9) @ rotate_vector(k, 12)
assert torch.allclose(score, shifted_score, atol=1e-5)

## 11. GQA, MLA, and efficient attention
**Grouped-query attention:** Use $H_q$ query heads and $H_{kv}$ key/value heads, with $H_q$ divisible by $H_{kv}$. Each group of query heads shares one K/V head. The conceptual expansion below is convenient for understanding; optimized kernels can avoid materializing repeated K/V tensors.

**Multi-head latent attention:** Compress each token's hidden vector into $c_t=W_{down}x_t$, then reconstruct head-specific keys and values through learned up-projections. Its cache can retain a lower-dimensional latent. RoPE needs special handling, often through a separate positional key component. The repository's MLA implementation includes that machinery.

**Efficient attention:** `scaled_dot_product_attention` computes the same operation using an available backend. FlashAttention availability depends on device, dtype, and backend support. It reduces intermediate memory traffic; exact full attention still has quadratic pairwise work in sequence length.

**Exercise:** For $H_q=8,H_{kv}=2$, identify which query heads share a K/V head. Estimate K/V storage relative to ordinary eight-head attention.

[Repository attention and cache details](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/docs/modern/attention.md)

In [ ]:
query = torch.randn(B, 4, T, 8)
key_small = torch.randn(B, 2, T, 8)
value_small = torch.randn(B, 2, T, 8)
key = key_small.repeat_interleave(2, dim=1)
value = value_small.repeat_interleave(2, dim=1)
manual, _ = causal_attention(query, key, value)
optimized = F.scaled_dot_product_attention(query, key, value, dropout_p=0.0, is_causal=True)
assert torch.allclose(manual, optimized, atol=1e-5)
# A minimal latent reconstruction illustrates MLA's compression step.
latent = nn.Linear(C, 8, bias=False)(x)
reconstructed_keys = nn.Linear(8, 4 * 8, bias=False)(latent)
assert reconstructed_keys.shape == (B, T, 32)

## 12. KV caching
During generation, past keys and values can be reused. A new query attends to the stored prefix plus its own key/value. A conventional cache stores approximately $2BTH_{kv}d$ numbers per layer.

Caching avoids recomputing old hidden states and projections. Each new query still attends over the cached sequence, so attention work per decoding token grows with the prefix length. Across a growing generated sequence, total attention work remains quadratic.

For cached chunks, a mask must use absolute query positions. For one query at the end of a complete prefix, every cached key is allowed. Blindly using a square-prefix causal flag with unequal Q/K lengths can select the wrong positions.

**Experiment:** The last row from a full causal pass should match attention from only the last query against all cached keys.

In [ ]:
full, _ = causal_attention(query, key, value)
cached_last = F.scaled_dot_product_attention(query[:, :, -1:], key, value, dropout_p=0.0, is_causal=False)
assert torch.allclose(full[:, :, -1:], cached_last, atol=1e-5)
print("Single-query cache attention agrees with the full pass.")

## 13. Mixture of Experts and the remaining modern options
A router scores experts for each token. Select top-$k$ experts and combine their outputs:
$$y_t=\sum_{e\in\mathrm{TopK}(r(x_t))}p_{t,e}E_e(x_t).$$
This increases total model capacity while evaluating a subset of experts per token. Routing overhead, imbalance, and expert utilization affect actual efficiency. The code below is a small top-2 routing demonstration; it evaluates every expert to keep the expression easy to inspect. Production sparse dispatch evaluates selected experts.

The linked modern documentation also describes QK normalization, optional output gates, sliding-window masks, tied input/output embeddings, bias-free projections, and depth-scaled initialization. These are configuration-dependent additions. Tying assigns the same parameter matrix to input embedding and output projection, saving one $VC$ matrix. Sliding windows restrict allowed keys to a recent interval.

[Repository MoE](https://github.com/FareedKhan-dev/train-llm-from-scratch/blob/main/docs/modern/moe.md)

**Exercise:** Set top-$k$ to one. How do the weights change? Why does this dense demonstration fail to realize sparse computation savings?

In [ ]:
experts = nn.ModuleList([SwiGLU(C, 88) for _ in range(4)])
router = nn.Linear(C, 4, bias=False)
scores, selected = router(x).topk(2, dim=-1)
weights = scores.softmax(-1)
all_outputs = torch.stack([expert(x) for expert in experts], dim=-2)
selected_outputs = all_outputs.gather(-2, selected.unsqueeze(-1).expand(B, T, 2, C))
mixture = (selected_outputs * weights.unsqueeze(-1)).sum(-2)
assert mixture.shape == (B, T, C)
assert torch.allclose(weights.sum(-1), torch.ones(B, T))

## 14. Independent implementation assignment
Create your own classes in a fresh notebook section. Use the definitions above as references after your first attempt.

| Task | Required result | Points |
|---|---|---|
| Derive MLP and attention shapes | Annotate every intermediate | 15 |
| Implement one attention head | Uniform-score example and row sums pass | 20 |
| Assemble multi-head attention and block | Parameter count and identity residual pass | 20 |
| Assemble full Transformer | Logits, loss, backward, and causality checks pass | 25 |
| Explain modern alternatives | Derive RoPE invariance and cache storage | 20 |

**Written questions**
1. Why can MLP output at position 2 stay unchanged when position 5 changes?
2. Why can attention output at position 5 change when position 2 changes?
3. Why is the diagonal included in the causal mask when predicting the next token?
4. Why must softmax normalize the key axis?
5. Why do learned position embeddings impose a configured maximum length?
6. Which memory term shrinks when reducing K/V heads?

**Answer guide:** The MLP acts independently at each position. Causal attention at 5 includes key 2. Input position $t$ contains the current token and predicts $t+1$, so attending to itself is permitted. Each query needs a distribution over its available keys. The learned position table has finitely many rows. GQA reduces cached K/V storage, proportional to $H_{kv}$.

## 15. Completion checklist and repository navigation
- Explain the complete path from Lab 1 IDs through embeddings, blocks, final normalization, logits, and cross-entropy.
- Pass the numerical attention, shape, residual, gradient, causality, and parameter-count checks.
- Explain every modern replacement listed in Step 2.
- Open `src/models/modern/` in this order: `norm.py`, `rope.py`, `mlp.py`, `attention.py`, `kv_cache.py`, `moe.py`, `block.py`, `model.py`.
- Continue to Step 3 only after you can explain why a model with a broken causal mask can appear to learn deceptively quickly.

This notebook covers model construction. Sustained optimization, evaluation curves, and checkpointing belong to the next lab. Numerical checks are included for you to execute; the authoring environment did not have PyTorch installed.
